# AME 5053 — Principles of Natural Language Processing Lab
## Word Embeddings and Cosine Similarity

**Suggested duration:** 3 hours

### Learning outcomes
By the end of this lab, you should be able to:

1. compute and interpret cosine similarity between vectors;
2. use TF–IDF vectors to rank documents for a query;
3. train small Word2Vec models using **CBOW** and **Skip-gram**;
4. inspect nearest neighbours in a learned embedding space;
5. verify that Word2Vec similarity is based on cosine similarity;
6. visualise learned embeddings in two dimensions using PCA;
7. connect **local Word2Vec training examples** with **global co-occurrence evidence** used by GloVe.

### Suggested workflow
Work through the notebook in order. Do not focus only on obtaining a number. At each stage, interpret what the number or plot means.

> **Important:** The Word2Vec corpus in this lab is intentionally small and synthetic so that the behaviour is easy to inspect in class. Real embedding models are trained on far larger and more varied corpora.

In [ ]:
# Core imports
import sys, subprocess, importlib.util, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# Install gensim only if the environment does not already contain it.
if importlib.util.find_spec("gensim") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gensim"])

from gensim.models import Word2Vec

np.set_printoptions(precision=3, suppress=True)
print("Setup complete.")

# Activity 1 — Cosine similarity as geometry

For two vectors $\vec a$ and $\vec b$,

$\cos(\vec a,\vec b)
=
\frac{\vec a\cdot\vec b}
{\|\vec a\|\,\|\vec b\|}$

Cosine similarity measures **directional alignment**, not simply vector length.

Use these vectors:

$\
A=[1,1],\qquad B=[2,2],\qquad C=[1,-1].
\$

Before running code, predict:

- Which pair should have cosine similarity \(1\)?
- Which pair should have cosine similarity \(0\)?

In [ ]:
A = np.array([1.0, 1.0])
B = np.array([2.0, 2.0])
C = np.array([1.0, -1.0])

def cosine_manual(a, b):
    # TODO 1: compute dot product
    dot = ...
    # TODO 2: compute product of vector norms
    denom = ...
    # TODO 3: return the cosine similarity
    return ...

print("cos(A,B) =", cosine_manual(A, B))
print("cos(A,C) =", cosine_manual(A, C))
print("cos(A,3A) =", cosine_manual(A, 3*A))

In [ ]:
# Visualise the three vectors.
origin = np.zeros(2)

plt.figure(figsize=(6, 6))
for name, v in [("A", A), ("B", B), ("C", C)]:
    plt.quiver(*origin, v[0], v[1], angles="xy", scale_units="xy", scale=1)
    plt.text(v[0] + 0.08, v[1] + 0.08, name, fontsize=12)

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlim(-2.5, 3)
plt.ylim(-2.5, 3)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.title("Cosine similarity depends on direction")
plt.grid(alpha=0.25)
plt.show()

**Checkpoint:** Explain in one sentence why multiplying a vector by 3 does not change its cosine similarity with the original vector.

# Activity 2 — From a query to ranked documents with TF–IDF + cosine

We now use a small restaurant corpus. The query is

> **affordable vegetarian food near beach**

The software library below uses a standard scikit-learn TF–IDF convention. Its exact numerical TF–IDF weights may differ from hand calculations that use a different logarithm or smoothing convention. The important idea is unchanged:

$$
\text{text} \rightarrow \text{TF–IDF vector} \rightarrow \text{cosine similarity} \rightarrow \text{ranking}.
$$

In [ ]:
documents = {
    "D1": "Coastal Cafe near Malpe Beach serves affordable vegetarian food and vegetarian snacks.",
    "D2": "Green Garden serves vegetarian food in the city centre.",
    "D3": "Ocean View is near the beach and serves seafood.",
    "D4": "Budget Cafe offers affordable snacks near the bus station."
}
query = "affordable vegetarian food near beach"

pd.DataFrame({"Document": documents.keys(), "Text": documents.values()})

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")

# Fit on documents + query so that all vectors use the same feature space.
all_text = list(documents.values()) + [query]
X = vectorizer.fit_transform(all_text)

document_vectors = X[:-1]
query_vector = X[-1]

# TODO: compute cosine similarity between query_vector and every document vector.
scores = ...

ranking = (
    pd.DataFrame({"Document": list(documents.keys()), "Cosine similarity": scores})
    .sort_values("Cosine similarity", ascending=False)
    .reset_index(drop=True)
)

ranking

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(ranking["Document"], ranking["Cosine similarity"])
plt.ylim(0, 1)
plt.xlabel("Document")
plt.ylabel("Cosine similarity to query")
plt.title("TF–IDF document ranking")
plt.show()

### Practice task
Change the query to:

> **seafood near beach**

Run the vectorisation and ranking again.

**Questions**
1. Which document becomes the strongest match?
2. Why?
3. Does cosine similarity understand that *seafood* and *vegetarian food* are semantically related types of food, or does it mainly depend on lexical overlap here?

In [ ]:
new_query = "seafood near beach"

# TODO: refit the vectorizer on documents + new_query, calculate cosine scores, and rank the documents.
# Write your code below.

# Activity 3 — Train word embeddings with Word2Vec

We will train two models on the same small corpus:

- **CBOW**: surrounding context $\rightarrow$ centre word
- **Skip-gram**: centre word $\rightarrow$ surrounding context

The corpus deliberately repeats a few semantic patterns such as **tea–coffee**, **doctor–nurse**, **car–bus**, and **apple–banana**. This makes the learned relationships visible within a short lab.

The repetition below is only for a stable classroom demonstration; it is not a substitute for a large natural corpus.

In [ ]:
base_sentences = [
    "people drink hot tea in a cup",
    "people drink hot coffee in a cup",
    "tea is served hot in the morning",
    "coffee is served hot in the morning",
    "she ordered tea at the cafe",
    "she ordered coffee at the cafe",
    "tea and coffee are common beverages",
    "the cup contains tea",
    "the cup contains coffee",
    "fresh tea tastes good",
    "fresh coffee tastes good",

    "doctor treats patient in hospital",
    "nurse cares for patient in hospital",
    "doctor and nurse work in hospital",
    "patient visited doctor for treatment",
    "patient spoke with nurse",
    "hospital staff includes doctor and nurse",

    "car travels on road",
    "bus travels on road",
    "driver drives car on road",
    "driver drives bus on road",
    "car and bus are vehicles",
    "road traffic includes car and bus",

    "apple is a fruit",
    "banana is a fruit",
    "apple and banana are fresh fruit",
    "people eat apple for breakfast",
    "people eat banana for breakfast",
    "fruit includes apple and banana"
]

tokenised = [re.findall(r"[a-z]+", s.lower()) for s in base_sentences]

# Repeat and shuffle deterministically to make the tiny demonstration stable.
training_corpus = tokenised * 40
random.Random(7).shuffle(training_corpus)

print("Number of training sentences:", len(training_corpus))
print("Example tokenised sentence:", training_corpus[0])

In [ ]:
# TODO: train one CBOW model and one Skip-gram model.
# Hints:
#   sg=0  -> CBOW
#   sg=1  -> Skip-gram
# Keep the remaining settings the same so the comparison is fair.

cbow_model = Word2Vec(
    sentences=training_corpus,
    vector_size=40,
    window=3,
    min_count=1,
    workers=1,
    sg=...,
    seed=7,
    epochs=120
)

skipgram_model = Word2Vec(
    sentences=training_corpus,
    vector_size=40,
    window=3,
    min_count=1,
    workers=1,
    sg=...,
    seed=7,
    epochs=120
)

print("Vocabulary size:", len(skipgram_model.wv))
print("Shape of the vector for 'coffee':", skipgram_model.wv["coffee"].shape)

## Inspect the learned vectors

A learned embedding dimension does **not** normally have a simple label such as “beverage” or “vehicle”. Meaning is distributed across many coordinates.

Run the next cell only to see what a dense vector looks like.

In [ ]:
coffee_vec = skipgram_model.wv["coffee"]
print("First 10 coordinates of the 40-dimensional 'coffee' vector:")
print(coffee_vec[:10])

# Activity 4 — Verify that embedding similarity is cosine similarity

Gensim's `similarity(word1, word2)` compares the learned word vectors using cosine similarity.

Verify this manually for **tea** and **coffee**.

In [ ]:
tea_vec = skipgram_model.wv["tea"]
coffee_vec = skipgram_model.wv["coffee"]

# TODO: calculate cosine similarity manually.
manual_similarity = ...

gensim_similarity = skipgram_model.wv.similarity("tea", "coffee")

print("Manual cosine:", manual_similarity)
print("Gensim cosine:", gensim_similarity)

# Activity 5 — Nearest neighbours in embedding space

For each word below, inspect the nearest words learned by Skip-gram.

Before running the code, predict what you expect for:

- `tea`
- `car`
- `apple`
- `doctor`

In [ ]:
for word in ["tea", "car", "apple", "doctor"]:
    print(f"\nNearest words to '{word}':")
    for neighbour, score in skipgram_model.wv.most_similar(word, topn=5):
        print(f"  {neighbour:12s} {score:.3f}")

## Compare selected pairs

A high cosine value means that the learned vectors point in similar directions. It does **not** prove that the words are synonyms.

In [ ]:
pairs = [
    ("tea", "coffee"),
    ("car", "bus"),
    ("apple", "banana"),
    ("doctor", "nurse"),
    ("tea", "car"),
    ("apple", "hospital")
]

rows = []
for a, b in pairs:
    rows.append({
        "Pair": f"{a} – {b}",
        "CBOW cosine": cbow_model.wv.similarity(a, b),
        "Skip-gram cosine": skipgram_model.wv.similarity(a, b)
    })

pd.DataFrame(rows)

# Activity 6 — Visualise embeddings with PCA

Word2Vec vectors here have 40 dimensions, so we cannot directly draw them. PCA projects them to two dimensions for visual inspection.

**Important:** PCA is only a projection. Distances in the 2-D plot are an approximation to structure in the original embedding space.

In [ ]:
selected_words = [
    "tea", "coffee", "cup",
    "doctor", "nurse", "hospital",
    "car", "bus", "road",
    "apple", "banana", "fruit"
]

vectors = np.vstack([skipgram_model.wv[w] for w in selected_words])
coords = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(9, 7))
plt.scatter(coords[:, 0], coords[:, 1], s=70)

for word, (x, y) in zip(selected_words, coords):
    plt.text(x + 0.02, y + 0.02, word, fontsize=11)

plt.xlabel("PCA dimension 1")
plt.ylabel("PCA dimension 2")
plt.title("2-D projection of learned Skip-gram embeddings")
plt.grid(alpha=0.2)
plt.show()

### Interpret the plot
Answer in 3–5 sentences:

1. Which pairs appear close?
2. Do the clusters broadly agree with the contexts in the training corpus?
3. Why should we avoid claiming that the 2-D PCA plot is the complete embedding space?

# Activity 7 — From local windows to global co-occurrence evidence

Word2Vec and GloVe both use neighbouring context, but they organise the evidence differently.

- **Word2Vec:** local windows become prediction examples.
- **GloVe:** local windows are accumulated into a corpus-wide word–context co-occurrence matrix before the dense vectors are learned.

We will build the **co-occurrence matrix** used to motivate GloVe. We are **not implementing the full GloVe optimiser** in this lab.

In [ ]:
from collections import Counter, defaultdict

def build_cooccurrence(sentences, window=2):
    vocab = sorted({w for sent in sentences for w in sent})
    idx = {w: i for i, w in enumerate(vocab)}
    matrix = np.zeros((len(vocab), len(vocab)), dtype=float)

    for sent in sentences:
        for i, target in enumerate(sent):
            left = max(0, i - window)
            right = min(len(sent), i + window + 1)
            for j in range(left, right):
                if i == j:
                    continue
                context = sent[j]
                matrix[idx[target], idx[context]] += 1

    return vocab, idx, matrix

vocab, word_to_idx, cooc = build_cooccurrence(tokenised, window=2)
cooc_df = pd.DataFrame(cooc, index=vocab, columns=vocab)

In [ ]:
# TODO: inspect these target words against these context words.
targets = ["tea", "coffee", "car"]
contexts = ["drink", "hot", "cup", "road"]

subset = ...
subset

In [ ]:
# Compare the complete co-occurrence profiles.
tea_profile = cooc[word_to_idx["tea"]]
coffee_profile = cooc[word_to_idx["coffee"]]
car_profile = cooc[word_to_idx["car"]]

# TODO: calculate cosine(tea, coffee) and cosine(tea, car)
print("Co-occurrence cosine tea–coffee:", ...)
print("Co-occurrence cosine tea–car:", ...)

In [ ]:
plt.figure(figsize=(7, 4))
plt.imshow(subset.values, aspect="auto")
plt.xticks(range(len(contexts)), contexts)
plt.yticks(range(len(targets)), targets)
plt.colorbar(label="Co-occurrence count")
plt.title("Selected word–context co-occurrence counts")
plt.xlabel("Context word")
plt.ylabel("Target word")
plt.show()

# Final interpretation task

Write a short answer (about 150–200 words) addressing all four points:

1. Why does cosine similarity suit both TF–IDF vectors and dense word embeddings?
2. What did the Word2Vec model learn from the local context examples in this lab?
3. How is the co-occurrence evidence prepared differently for GloVe?
4. Why should a high cosine similarity between two embeddings be interpreted as **distributional relatedness**, not automatically as synonymy?

### Optional advanced task
Explain, in about one page:

> **How is GloVe trained differently from Word2Vec, and what happens after the word vectors have been learned?**

Use one example word pair such as **coffee–drink**. Distinguish clearly between:
- a Skip-gram training pair, and
- an entry $X_{ij}$ in a GloVe-style co-occurrence matrix.